In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "07-application-agent-framework/retrieval-rag/rag-from-scratch/solutions")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 02 · Chunking

Whole-document chunks (nb 01) waste the context window and blur one vector
across many topics. But naive splitting has the opposite failure: it cuts
sentences in half and strips away the heading that gave them meaning.

The key move: **decouple what you match on from what you show the model**.


In [ ]:
# --- setup: make `import ragkit` work from notebooks/ or solutions/ ---
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
from ragkit.corpus import load_documents, load_corpus, load_qrels, tokenize
from ragkit.embed import get_embedder
from ragkit import llm

In [ ]:
docs = load_documents()
sample = docs["security-access"]
print(sample)

### Baseline — fixed-size windows

Split into fixed word windows with a little overlap. Simple, structure-blind.


In [ ]:
def fixed_size_chunks(doc_id, text, size=40, overlap=10):
    words = text.split()
    step = max(1, size - overlap)
    out = []
    for i in range(0, len(words), step):
        window = words[i:i + size]
        if not window:
            break
        out.append(" ".join(window))
        if i + size >= len(words):
            break
    return out

fx = fixed_size_chunks("security-access", sample)
print(f"{len(fx)} chunks. Notice a chunk can start mid-sentence:\n")
print("-", fx[1][:160], "...")

### Exercise 1 — structure-aware chunks

Markdown already tells us the boundaries. Split on `##` headings, and **prepend
the doc title + heading** to each chunk so it is self-describing (a chunk that
says *"Access Control Standard (SEC-011) > Production Access"* retrieves far better than a
bare paragraph that starts with "Access to production...").

Return a list of `(section_heading, chunk_text)` where `chunk_text` starts with
the header line `"<title> > <heading>"`.


In [ ]:
def structure_aware_chunks(doc_id, text):
    lines = text.splitlines()
    title = lines[0].lstrip("# ").strip()      # first line is "# Title"
    out, heading, buf = [], None, []

    def flush():
        if heading and buf:
            body = "\n".join(buf).strip()
            if body:
                out.append((heading, f"{title} > {heading}\n{body}"))

    for line in lines:
        if line.startswith("## "):
            flush()
            heading, buf = line[3:].strip(), []
        elif not line.startswith("# "):
            buf.append(line)
    flush()
    return out

sec = structure_aware_chunks("security-access", sample)
headings = [h for h, _ in sec]
assert "Production Access" in headings
prod = next(t for h, t in sec if h == "Production Access")
assert prod.startswith("Access Control Standard")        # doc title kept as header
assert "> Production Access" in prod                      # heading in the header line
assert "VPN" in prod                                                     # body preserved
print("sections:", headings)
print("\n", prod)

### Small-to-big (sentence-window)

Precision *and* context: match on a small unit (a sentence), but **return** its
parent section. You search with fine granularity and hand the model enough
surrounding text to answer. The reference `sentence_chunks` implements this —
we import it rather than rebuild it.


In [ ]:
from ragkit.reference import sentence_chunks
small, parent_of = sentence_chunks("security-access", sample)
print(f"{len(small)} sentence-level match units -> {len(set(p.chunk_id for p in parent_of.values()))} parent sections\n")
s = small[3]
print("match unit :", s.text)
print("return this:", parent_of[s.chunk_id].text[:120], "...")

**Takeaway.** Structure-aware chunks with prepended headers are the highest-ROI
default for document corpora. Small-to-big adds precision when sections are
long. We'll quantify the difference in notebook 05.
